<a href="https://colab.research.google.com/github/PalakPrajapati346/WORKSHOP-2/blob/main/Best.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install lightgbm pandas numpy scikit-learn pygeohash

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 1.7 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import pygeohash as pgh
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
import warnings

In [3]:
warnings.filterwarnings('ignore')

In [6]:
print("Loading datasets...")
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

Loading datasets...


In [13]:
target_col = 'demand'
train_len = len(train_df)
y_train = train_df[target_col].values

# Combine for unified feature engineering
combined = pd.concat([train_df.drop(columns=[target_col]), test_df], axis=0).reset_index(drop=True)

# 2. FEATURE ENGINEERING
print("Engineering features...")

# A. Decode Geohash to Latitude/Longitude
combined['lat'] = combined['geohash'].apply(lambda x: pgh.decode(x)[0] if pd.notnull(x) else np.nan)
combined['lon'] = combined['geohash'].apply(lambda x: pgh.decode(x)[1] if pd.notnull(x) else np.nan)

# B. Parse Temporal Attributes
combined['timestamp'] = pd.to_datetime(combined['timestamp'], format='%H:%M') # adjusting format if needed
combined['hour'] = combined['timestamp'].dt.hour
combined['minute'] = combined['timestamp'].dt.minute
combined['time_fraction'] = combined['hour'] + combined['minute'] / 60.0

# Cyclic transformation for time of day
combined['sin_time'] = np.sin(2 * np.pi * combined['time_fraction'] / 24.0)
combined['cos_time'] = np.cos(2 * np.pi * combined['time_fraction'] / 24.0)

# Day features
combined['day_of_week'] = combined['day'] % 7  # Assuming 'day' is a sequential integer counter

# C. Categorical Encodings
# C. Categorical Encodings
# Added 'LargeVehicles' and 'Landmarks' to the categorical list to fix the ValueError
categorical_cols = ['RoadType', 'Weather', 'geohash', 'LargeVehicles', 'Landmarks']

for col in categorical_cols:
    # Ensure any unexpected missing values are handled, then cast to string before category
    combined[col] = combined[col].fillna('Unknown').astype(str).astype('category')

# Drop raw components that aren't useful as numbers anymore
combined.drop(columns=['timestamp'], errors='ignore', inplace=True)

# Split back to Train and Test
X_train = combined.iloc[:train_len].copy()
X_test = combined.iloc[train_len:].copy()

# D. Out-of-fold Target Encoding for Geohash (Prevents Data Leakage)
X_train['geohash_mean_demand'] = np.nan
X_test['geohash_mean_demand'] = np.nan

Engineering features...


In [14]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
global_mean = y_train.mean()

In [15]:
for train_idx, val_idx in kf.split(X_train):
    # Compute mean on the training fold
    fold_train = X_train.iloc[train_idx]
    geohash_means = y_train[train_idx]
    temp_df = pd.DataFrame({'geohash': fold_train['geohash'], 'target': geohash_means})
    means = temp_df.groupby('geohash')['target'].mean()

    # Map to validation fold
    X_train.iloc[val_idx, X_train.columns.get_loc('geohash_mean_demand')] = X_train.iloc[val_idx]['geohash'].map(means)

In [16]:
X_train['geohash_mean_demand'].fillna(global_mean, inplace=True)

full_train_df = pd.DataFrame({'geohash': X_train['geohash'], 'target': y_train})
test_means = full_train_df.groupby('geohash')['target'].mean()
X_test['geohash_mean_demand'] = X_test['geohash'].map(test_means).fillna(global_mean)

# Drop Index column from feature matrices
features_to_drop = ['Index']
X_train_features = X_train.drop(columns=features_to_drop)
X_test_features = X_test.drop(columns=features_to_drop)

In [17]:
lgb_params = {
    'objective': 'regression',
    'metric': 'rmse',                  # Optimizing RMSE acts as a proxy optimizer for R2
    'boosting_type': 'gbdt',
    'n_estimators': 3000,              # Large number paired with early stopping
    'learning_rate': 0.03,             # Lower step-size for better convergence
    'num_leaves': 63,                  # High complexity but guarded by regularization
    'max_depth': -1,
    'min_child_samples': 20,
    'subsample': 0.8,                  # Row bagging to prevent overfitting
    'colsample_bytree': 0.8,           # Feature bagging
    'reg_alpha': 0.1,                  # L1 regularization
    'reg_lambda': 1.0,                 # L2 regularization
    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1
}

In [18]:
print("Starting Cross-Validated Training...")
oof_predictions = np.zeros(len(X_train))
test_predictions = np.zeros(len(X_test))

for fold, (train_idx, val_idx) in enumerate(kf.split(X_train_features, y_train)):
    print(f"--- Training Fold {fold + 1} ---")

    X_tr, y_tr = X_train_features.iloc[train_idx], y_train[train_idx]
    X_va, y_va = X_train_features.iloc[val_idx], y_train[val_idx]

    model = lgb.LGBMRegressor(**lgb_params)

    model.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False), lgb.log_evaluation(0)]
    )

    # Store predictions
    oof_predictions[val_idx] = model.predict(X_va)
    test_predictions += model.predict(X_test_features) / kf.n_splits

Starting Cross-Validated Training...
--- Training Fold 1 ---
--- Training Fold 2 ---
--- Training Fold 3 ---
--- Training Fold 4 ---
--- Training Fold 5 ---


In [19]:
cv_r2 = r2_score(y_train, oof_predictions)
final_score = max(0, 100 * cv_r2)
print(f"\n[LOCAL EVALUATION] Out-of-fold R2 Score: {cv_r2:.5f}")
print(f"[LOCAL EVALUATION] Anticipated Hackathon Score: {final_score:.2f}")


[LOCAL EVALUATION] Out-of-fold R2 Score: 0.96049
[LOCAL EVALUATION] Anticipated Hackathon Score: 96.05


In [21]:
submission = pd.DataFrame({
    'Index': test_df['Index'],
    'demand': test_predictions
})

submission.to_csv("submission.csv", index=False)
print("Submission file 'submission.csv' generated successfully with shapes:", submission.shape)

Submission file 'submission.csv' generated successfully with shapes: (41778, 2)
